# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

In [66]:
import polars as pl

In [67]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [68]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")
scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
1198,9634,0,"""VUSAPWT2B1""",0.4375,2,1,1,null,null,"""TNFSF14""","""CLDN19""",null,null,"""ENSG00000164007""",null,null,null,null,"""CLDN19"""
5793,4062,201426,"""VUSAPWT2B1""",0.875,2,1,1,1.0,1.0,"""MS4A3""","""PLP1""","""Thr147Pro""","""439A>C""","""ENSG00000123560""","""CEGS2""","""1_Pathogenic""",null,null,"""PLP1_Thr147Pro"""
6490,2920,214409,"""VUSAPWT2B1""",0.4375,2,1,1,1.0,1.0,"""PIAS2""","""GFAP""","""Glu205Lys""","""613G>A""","""ENSG00000131095""","""CEGS2""","""3_Conflicting""",null,1.0,"""GFAP_Glu205Lys"""
4621,4723,201744,"""VUSAPWT1B1""",1.0,1,1,1,1.0,2.0,"""KXD1""","""TPM1""","""Asp230Asn""","""688G>A""","""ENSG00000140416""","""CEGS2""","""1_Pathogenic""",null,2.0,"""TPM1_Asp230Asn"""
52634,3535,201190,"""VUSAPWT6B1""",0.0625,6,1,1,1.0,1.0,"""CPLX4""","""EMD""","""Met1Val""","""1A>G""","""ENSG00000102119""","""CEGS2""","""1_Pathogenic""",9.5000e-7,2.0,"""EMD_Met1Val"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4623,1256,214010,"""VUSAPWT1B2""",0.125,1,1,1,1.0,1.0,"""KLHL12""","""KLHL3""","""Cys164Phe""","""491G>T""","""ENSG00000146021""","""CEGS2""","""1_Pathogenic""",null,null,"""KLHL3_Cys164Phe"""
208,5066,0,"""VUSAPWT1B2""",0.5,1,1,1,null,null,"""ARRDC3""","""BAG3""",null,null,"""ENSG00000151929""",null,null,null,null,"""BAG3"""
56141,70497,3812,"""VUSAPWT1B1""",0.0625,1,1,1,1.0,1.0,"""KRT31""","""AGXT""","""Arg111Gln""","""332G>A""","""ENSG00000172482""","""Edgotyping3""","""1_Pathogenic""",0.000022,2.0,"""AGXT_Arg111Gln"""


In [69]:
scores_pd = scores.to_pandas()
scores_pd[
    (scores_pd.ad_orf_id == '12626.0') &
    (scores_pd.db_orf_id == '9905.0') #&
    # (scores_pd.db_mut_id == 215290)
    ]

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0


In [70]:
import os
import shutil

# Remove all files in subfolders edgotyping_profiles
profile_dirs = [
    # f"{outputs_dir}/edgotype_profiles/manual",
    f"{outputs_dir}/edgotype_profiles/auto_norm", 
    f"{outputs_dir}/edgotype_profiles/auto_summed"
]

for profile_dir in profile_dirs:
    if os.path.exists(profile_dir):
        # Remove all files in the directory
        for filename in os.listdir(profile_dir):
            file_path = os.path.join(profile_dir, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')
        print(f"Cleared all files from {profile_dir}")
    else:
        print(f"Directory {profile_dir} does not exist")

Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_norm
Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_summed


In [71]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

# print(scores)

In [72]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/manual/orf_id_{gene}.csv")

shape_summary = pl.DataFrame(shape_summary)

In [73]:
## Add a perturbation category for each edge to use for the N2H validation

# Extract WT scores
wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# Merge WT scores back onto the main df
scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# Compute allele/WT ratio
scores_with_wt = scores_with_wt.with_columns(
    (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
)

# Determine if the allele is perturbed - binary output
threshold = 0.5
scores_with_wt = scores_with_wt.with_columns(
    ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
)

# Build perturbation_classes expression
bins = [
    (1.0, float("inf"), "increased edge"),
    (0.8, 1.0, "similar"),
    (0.6, 0.8, "slightly perturbed"),
    (0.4, 0.6, "perturbed"),
    (0.2, 0.4, "highly perturbed"),
    (0.0, 0.2, "very highly perturbed"),
]

perturbation_expr = pl.when(pl.col("db_mut_id") == 0).then(pl.lit("WT"))

perturbation_expr = perturbation_expr.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

for lower, upper, label in bins:
    perturbation_expr = perturbation_expr.when((pl.col("ratio") > lower) & (pl.col("ratio") <= upper)).then(pl.lit(label))


perturbation_expr = perturbation_expr.otherwise(pl.lit("unclassified"))




scores_with_wt = scores_with_wt.with_columns(
    perturbation_expr.alias("perturbation_classes")
)

scores_with_perturbed = scores_with_wt.select("*")


pert2 = (pl.when(pl.col("db_mut_id") == 0).then(pl.lit('WT')))

pert2 = pert2.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

pert2 = pert2.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

pert2 = pert2.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

bins_2 = [
    (float("inf"), 1.0, "increased edge"),
    (1, 0.7, 'slightly perturbed'),
    (0.7, 0.3, 'perturbed'),
    (0.3, 0, 'highly perturbed')
]

for upper, lower, label in bins_2:
    pert2 = pert2.when(
        (pl.col("ratio") <= upper) &
        (pl.col("ratio") >  lower)
    ).then(pl.lit(label))

pert2 = pert2.otherwise(pl.lit("unclassified"))

scores_with_perturbed = scores_with_wt.with_columns(
    perturbation_classes_2 = pert2.alias("perturbation_classes_2")
)



cols_to_keep = ['ad_orf_id',
 'db_orf_id',
 'db_mut_id',
 'normalized_score',
 'ad_symbol',
 'symbol',
 'clinvar_clnsig_clean',
 'normalized_score_wt',
 'ratio',
 'perturbed',
 'perturbation_classes',
 "perturbation_classes_2"]

scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")

## Examine profile characteristics

In [74]:
# Number of profiled alleles with more than 5 interactors
hq_orfs = shape_summary.filter(pl.col("n_interactors") >= 5)
num_orfs = hq_orfs.shape[0]
print(num_orfs)

print(hq_orfs.select(pl.col("n_alleles")).sum() - num_orfs)

25
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 255       │
└───────────┘


In [75]:
# Total number of profiled alleles
prof_genes = shape_summary.shape[0]
print(prof_genes)

print(shape_summary.select(pl.col("n_alleles")).sum() - prof_genes)

84
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 659       │
└───────────┘


# Automated

In [76]:
scores = pl.read_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")
scores = scores.drop_nulls(subset=["normalized_score"])

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,f64,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
13850,71001,205676,"""VUSAPWT6B1""",1674.504092,1788.335908,0.936348,6,1,1,1.0,1.0,"""CSN2""","""UBQLN2""","""Pro509Ser""","""1525C>T""","""ENSG00000188021""","""CEGS2""","""4_VUS""",0.000014,1.0,"""UBQLN2_Pro509Ser"""
13628,4062,0,"""VUSAPWT2B1""",683.974644,683.974644,1.0,2,1,1,null,null,"""KLRC1""","""PLP1""",null,null,"""ENSG00000123560""",null,null,null,null,"""PLP1"""
55351,5066,0,"""VUSAPWT1B2""",683.80819,683.80819,1.0,1,1,1,null,null,"""TP53BP2""","""BAG3""",null,null,"""ENSG00000151929""",null,null,null,null,"""BAG3"""
3818,4723,201725,"""VUSAPWT2B1""",49.858171,142.266014,0.350457,2,1,1,1.0,1.0,"""PSMC5""","""TPM1""","""Glu23Gln""","""67G>C""","""ENSG00000140416""","""CEGS2""","""4_VUS""",null,1.0,"""TPM1_Glu23Gln"""
6644,5723,0,"""VUSAPWT6B1""",1156.263767,1156.263767,1.0,6,1,1,null,null,"""BSCL2""","""SFTPC""",null,null,"""ENSG00000168484""",null,null,null,null,"""SFTPC"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
5894,2920,214386,"""VUSAPWT1B2""",1456.222581,1940.24581,0.750535,1,1,1,1.0,1.0,"""TFIP11""","""GFAP""","""Arg88Cys""","""262C>T""","""ENSG00000131095""","""CEGS2""","""1_Pathogenic""",null,2.0,"""GFAP_Arg88Cys"""
71963,2713,200856,"""VUSAPWT6B1""",1735.061619,2612.802714,0.664061,6,1,1,1.0,1.0,"""MBD3L1""","""MLH1""","""Asp304Gly""","""911A>G""","""ENSG00000076242""","""CEGS2""","""3_Conflicting""",6.8405e-7,1.0,"""MLH1_Asp304Gly"""
1863,52920,2686,"""VUSAPWT1B1""",1826.22882,1621.828469,1.126031,1,1,1,1.0,1.0,"""STAM2""","""LITAF""","""Pro39Ser""","""115C>T""","""ENSG00000189067""","""Edgotyping3""","""4_VUS""",null,1.0,"""LITAF_Pro39Ser"""


In [77]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_norm/orf_id_{gene}_auto_normalized.csv")

shape_summary = pl.DataFrame(shape_summary)

In [78]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="adjusted_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_summed/orf_id_{gene}_auto_summed.csv")

shape_summary = pl.DataFrame(shape_summary)